In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import time
import matplotlib.pyplot as plt
from functools import partial
import jax.lax as lax

# Set precision (A100/H100 Optimal: FP32). Ensure this matches your JAX installation.
# jax.config.update("jax_enable_x64", False)

# =========================================================================
# 1. CORE PHYSICS ENGINE (Reconstructed and Optimized from Logs)
# =========================================================================
# This section reconstructs the optimized engine developed during the investigation.

C0_SU3 = 0.125  # Correct Haar coefficient N/24

def su3_generators():
    # Gell-Mann matrices
    lam1 = jnp.array([[0,1,0],[1,0,0],[0,0,0]], jnp.complex64)
    lam2 = jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], jnp.complex64)
    lam3 = jnp.array([[1,0,0],[0,-1,0],[0,0,0]], jnp.complex64)
    lam4 = jnp.array([[0,0,1],[0,0,0],[1,0,0]], jnp.complex64)
    lam5 = jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], jnp.complex64)
    lam6 = jnp.array([[0,0,0],[0,0,1],[0,1,0]], jnp.complex64)
    lam7 = jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], jnp.complex64)
    lam8 = jnp.array([[1,0,0],[0,1,0],[0,0,-2]], jnp.complex64) / jnp.sqrt(3.0)
    # Anti-hermitian basis T_a = i * lambda_a / 2
    return 1j * jnp.stack([lam1,lam2,lam3,lam4,lam5,lam6,lam7,lam8], 0) / 2.0

# Global definition of generators
T_SU3 = su3_generators()

def su3_alg_from_vec(a):
    return jnp.einsum("...a,aij->...ij", a, T_SU3)

# Use checkpointing for memory efficiency during backpropagation (needed for HMC gradient)
@jax.checkpoint
def su3_exp_pade22(A):
    """Differentiable Padé [2/2] approximant."""
    I = jnp.eye(3, dtype=jnp.complex64)
    A2 = A @ A
    # Use precise float32 fraction for c2 if needed: 0.083333336
    c1, c2 = 0.5, 1/12.0
    Num = I + c1*A + c2*A2
    Den = I - c1*A + c2*A2
    return jnp.linalg.solve(Den, Num)

def build_links_factory(L):
    @jax.checkpoint
    def build_links(params):
        flat = params.reshape(-1,8)
        A = jax.vmap(su3_alg_from_vec)(flat)
        U = jax.vmap(su3_exp_pade22)(A)
        return U.reshape(L,L,L,L,4,3,3)
    return build_links

# Optimized Wilson Action (JIT compiled, no checkpointing for speed)
@jax.jit
def compute_plaquette_sum(U, beta):
    S = 0.0
    for mu in range(4):
        for nu in range(mu+1, 4):
            U_mu = U[..., mu, :, :]
            U_nu_shift = jnp.roll(U[..., nu, :, :], -1, axis=mu)
            U_mu_dag_shift = jnp.swapaxes(
                jnp.conjugate(jnp.roll(U[..., mu, :, :], -1, axis=nu)), -1, -2)
            U_nu_dag = jnp.swapaxes(jnp.conjugate(U[..., nu, :, :]), -1, -2)

            P = U_mu @ U_nu_shift @ U_mu_dag_shift @ U_nu_dag
            trP = jnp.real(jnp.einsum("...ii->...", P))
            S += jnp.sum(1.0 - trP/3.0)
    return beta * S

@jax.jit
def haar_mass(params, c0):
    flat = params.reshape(-1,8)
    def per(a):
        A = su3_alg_from_vec(a)
        # Tr(A_dagger @ A).
        return jnp.real(jnp.trace(A.conj().T @ A))
    return c0 * jax.vmap(per)(flat).sum()

def make_flat_funcs(L, beta, c0):
    build_links_L = build_links_factory(L)
    # Determine dtype based on JAX configuration
    dtype = jnp.float32 if not jax.config.read("jax_enable_x64") else jnp.float64

    @jax.jit
    def flat_action(theta):
        # Ensure theta is cast to the working dtype if necessary
        theta = theta.astype(dtype)
        params = theta.reshape((L,L,L,L,4,8))
        U = build_links_L(params)
        S_W = compute_plaquette_sum(U, beta)
        S_H = haar_mass(params, c0)
        return S_W + S_H

    # Define the gradient function needed for HMC and Lanczos HVP
    flat_action_grad = jax.jit(jax.grad(flat_action))

    return flat_action, flat_action_grad, (L**4 * 4 * 8)

# =========================================================================
# 2. LANCZOS EIGENSOLVER
# =========================================================================

def hvp(flat_action_grad, theta, v):
    """Hessian-vector product using JVP of the gradient."""
    # We use the gradient function directly for JVP
    _, hv = jax.jvp(flat_action_grad, (theta,), (v,))
    return hv

def lanczos_min(flat_action_grad, theta, k=25, seed=0):
    """Estimate lambda_min(Hessian) via Lanczos."""
    key = jax.random.PRNGKey(seed)
    n = theta.shape[0]

    # Ensure v0 dtype matches theta dtype (important for JAX/XLA)
    v0 = jax.random.normal(key, (n,), dtype=theta.dtype)
    v0 /= jnp.linalg.norm(v0)

    def step(carry, _):
        v_prev, v, beta_prev = carry
        w = hvp(flat_action_grad, theta, v)
        w -= beta_prev * v_prev
        alpha = jnp.dot(w, v)
        w -= alpha * v
        beta = jnp.linalg.norm(w)
        # Add small epsilon for numerical stability
        v_next = w / (beta + 1e-9)
        return (v, v_next, beta), (alpha, beta)

    # Use lax.scan for efficient JIT compilation
    # Ensure initial beta_prev is the correct dtype
    init_beta = jnp.array(0.0, dtype=theta.dtype)
    (_, _, _), (alphas, betas) = lax.scan(
        step,
        init=(jnp.zeros_like(v0), v0, init_beta),
        xs=None,
        length=k
    )

    # Construct the tridiagonal matrix T
    alphas = jnp.array(alphas)
    betas = jnp.array(betas[:-1])
    T = jnp.diag(alphas) + jnp.diag(betas, 1) + jnp.diag(betas, -1)
    # Calculate eigenvalues of T
    return float(jnp.linalg.eigvalsh(T)[0])

# =========================================================================
# 3. HYBRID MONTE CARLO (HMC) SAMPLER
# =========================================================================

def kinetic_energy(p):
    """K(p) = 0.5 * p^T p."""
    return 0.5 * jnp.sum(p**2)

# JIT the integrator, marking the gradient function (index 1) and n_steps (index 4) as static
@partial(jax.jit, static_argnums=(1, 4))
def leapfrog_integrator(theta, flat_action_grad, p, dt, n_steps):
    """Symplectic (Leapfrog) integrator."""
    theta_new = theta
    p_new = p

    # Initial half-step (Kick)
    p_new -= 0.5 * dt * flat_action_grad(theta_new)

    # Full steps (Drift-Kick)
    # A standard for loop is acceptable here as n_steps is marked static (JAX unrolls it)
    for _ in range(n_steps - 1):
        theta_new += dt * p_new
        p_new -= dt * flat_action_grad(theta_new)

    # Final full step (Drift)
    theta_new += dt * p_new

    # Final half-step (Kick)
    p_new -= 0.5 * dt * flat_action_grad(theta_new)

    return theta_new, p_new

# JIT the HMC step, marking functions and static integers as static
@partial(jax.jit, static_argnums=(1, 2, 5))
def hmc_update(key, flat_action, flat_action_grad, theta, dt, n_steps):
    """One HMC update."""
    key_p, key_accept = jax.random.split(key)

    # 1. Sample momenta (Gaussian)
    p = jax.random.normal(key_p, theta.shape, dtype=theta.dtype)

    # 2. Calculate initial Hamiltonian H = S(A) + K(p)
    H_initial = flat_action(theta) + kinetic_energy(p)

    # 3. Molecular Dynamics trajectory
    theta_prop, p_prop = leapfrog_integrator(
        theta, flat_action_grad, p, dt, n_steps
    )

    # 4. Calculate final Hamiltonian
    H_final = flat_action(theta_prop) + kinetic_energy(p_prop)

    # 5. Metropolis-Hastings acceptance
    delta_H = H_final - H_initial
    # Handle potential NaNs if integration becomes unstable (e.g., dt too large)
    delta_H = jnp.where(jnp.isnan(delta_H), jnp.inf, delta_H)

    acceptance_prob = jnp.minimum(1.0, jnp.exp(-delta_H))

    # Accept or reject
    accept = jax.random.uniform(key_accept) < acceptance_prob
    theta_new = jnp.where(accept, theta_prop, theta)

    return theta_new, acceptance_prob, accept

# =========================================================================
# 4. ANALYSIS PIPELINE
# =========================================================================

def run_hmc_analysis(L, beta, c0=C0_SU3, n_therm=100, n_samples=500, dt=0.1, n_steps=10, seed=42):
    print(f"=== HMC MEASURE CONCENTRATION ANALYSIS (L={L}, Beta={beta:.2f}) ===")

    # Initialize engine
    flat_action, flat_action_grad, n_params = make_flat_funcs(L, beta, c0)

    # Determine dtype
    dtype = jnp.float32 if not jax.config.read("jax_enable_x64") else jnp.float64
    print(f"Parameters: {n_params}. Working Dtype: {dtype}")

    # Initialize configuration (start near the origin A=0)
    key = jax.random.PRNGKey(seed)
    key, subkey = jax.random.split(key)
    theta = 0.01 * jax.random.normal(subkey, (n_params,), dtype=dtype)

    # Thermalization (Bringing the system to equilibrium)
    print("Starting Thermalization...")
    t0 = time.time()
    avg_acc_prob = 0.0
    for i in range(n_therm):
        key, subkey = jax.random.split(key)
        theta, acc_prob, _ = hmc_update(subkey, flat_action, flat_action_grad, theta, dt, n_steps)
        avg_acc_prob += acc_prob
        if (i+1) % 20 == 0:
            S = flat_action(theta)
            print(f"  Therm Step {i+1}: Action={S:.4f}, AvgAccProb={avg_acc_prob/(i+1):.3f}")
    print(f"Thermalization done in {time.time()-t0:.2f}s")

    # Sampling and Measurement
    print("Starting Sampling and Measurement...")
    lambdas = []
    t0 = time.time()
    total_accepted = 0
    for i in range(n_samples):
        # Generate keys for HMC update and for Lanczos seed
        key, subkey_hmc, subkey_lanczos = jax.random.split(key, 3)

        # HMC Update
        theta, acc_prob, accepted = hmc_update(subkey_hmc, flat_action, flat_action_grad, theta, dt, n_steps)
        total_accepted += int(accepted)

        # Measure lambda_min using Lanczos on the generated configuration
        # We pass the gradient function (needed for HVP) to Lanczos
        lam = lanczos_min(flat_action_grad, theta, k=25, seed=int(subkey_lanczos[0]))
        lambdas.append(lam)

        if (i+1) % 50 == 0:
             S = flat_action(theta)
             print(f"  Sample {i+1}: Action={S:.4f}, AccProb={acc_prob:.3f}, Lambda_min={lam:+.6f}")

    print(f"Sampling done in {time.time()-t0:.2f}s")
    avg_acceptance_rate = total_accepted/n_samples
    print(f"Average Acceptance Rate: {avg_acceptance_rate:.2f}")
    if not (0.5 <= avg_acceptance_rate <= 0.85):
        print("NOTE: Acceptance rate is outside the typical optimal range (0.5-0.85). Consider tuning dt.")

    return np.array(lambdas)

def plot_hmc_results(lambdas, L, beta):
    plt.figure(figsize=(10, 6))
    plt.hist(lambdas, bins=50, density=True, alpha=0.7, color='navy')
    plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Gribov Horizon (λ=0)')
    plt.title(f'P(λ_min) for SU(3) L={L}, β={beta:.2f} (HMC Sampled)')
    plt.xlabel('λ_min')
    plt.ylabel('Probability Density')
    plt.legend()
    plt.grid(True, alpha=0.3)

    # Calculate probability of being outside the convex region
    prob_unstable = np.mean(lambdas < 0)
    mean_lambda = np.mean(lambdas)

    print(f"\n=== ANALYSIS RESULTS (L={L}, Beta={beta}) ===")
    print(f"Mean Lambda_min: {mean_lambda:+.6f}")
    print(f"P(Lambda_min < 0): {prob_unstable*100:.4f}%")

    if prob_unstable > 0.01:
        print("FINDING: Significant probability mass found outside the convex region.")
    else:
        print("FINDING: Measure is concentrated within the convex region.")

    plt.show()

# =========================================================================
# 5. EXECUTION BLOCK (HMC)
# =========================================================================

if __name__ == "__main__":
    # Configuration
    # Start with L=4 for testing/tuning. Scale up to L=6 or L=8 for production.
    L = 4
    # Choose a beta value in the weak coupling region (e.g., where static analysis showed instability)
    BETA = 3.0

    # HMC parameters (Tuning required)
    DT = 0.08      # Integration step size (tune this first)
    N_STEPS = 12   # Number of steps per trajectory
    N_THERM = 150
    N_SAMPLES = 500

    # To run the analysis, uncomment the following lines:
    # NOTE: Ensure JAX is installed and configured for your hardware (GPU highly recommended).

    # lambda_results = run_hmc_analysis(L=L, beta=BETA, n_therm=N_THERM,
    #                                   n_samples=N_SAMPLES, dt=DT, n_steps=N_STEPS)
    # plot_hmc_results(lambda_results, L, BETA)

    print("\nHMC Script loaded. Uncomment the execution lines in __main__ to run the analysis.")




HMC Script loaded. Uncomment the execution lines in __main__ to run the analysis.


In [2]:
# =========================================================================
# C_W ESTIMATION
# =========================================================================
# This script requires the Physics Engine functions (make_flat_funcs, hvp,
# T_SU3, etc.) defined in the HMC script above to be available in the scope.

def lanczos_opnorm(flat_action_grad, theta, k=30, seed=0):
    """
    Estimate the operator norm ||H|| (largest eigenvalue magnitude)
    of the Hessian via the Lanczos algorithm.
    """
    key = jax.random.PRNGKey(seed)
    n = theta.shape[0]
    v0 = jax.random.normal(key, (n,), dtype=theta.dtype)
    v0 /= jnp.linalg.norm(v0)

    # The Lanczos iteration step (identical to lanczos_min)
    def step(carry, _):
        v_prev, v, beta_prev = carry
        # HVP calculation using the provided gradient function
        w = hvp(flat_action_grad, theta, v)
        w -= beta_prev * v_prev
        alpha = jnp.dot(w, v)
        w -= alpha * v
        beta = jnp.linalg.norm(w)
        v_next = w / (beta + 1e-9)
        return (v, v_next, beta), (alpha, beta)

    # Run the iterations
    # Ensure initial beta_prev is the correct dtype
    init_beta = jnp.array(0.0, dtype=theta.dtype)
    init_val = (jnp.zeros_like(v0), v0, init_beta)
    (_, _, _), (alphas, betas) = lax.scan(step, init_val, None, length=k)

    # Construct the tridiagonal matrix T
    alphas = jnp.array(alphas)
    betas = jnp.array(betas[:-1])
    T = jnp.diag(alphas) + jnp.diag(betas, 1) + jnp.diag(betas, -1)

    # Compute eigenvalues of T
    eigs = jnp.linalg.eigvalsh(T)

    # The operator norm is the maximum absolute eigenvalue
    return float(jnp.max(jnp.abs(eigs)))

def estimate_CW(L, beta, n_samples=100, seed=42):
    """
    Estimate the constant C_W, where ||H_W(A)|| <= C_W * beta.
    We search across various field amplitudes to find the maximum norm.
    """
    # We need the Wilson-only action (c0=0.0)
    # Set c0=0.0 specifically for this estimation
    flat_action_W, flat_action_grad_W, n_params = make_flat_funcs(L, beta, c0=0.0)

    # Determine dtype
    dtype = jnp.float32 if not jax.config.read("jax_enable_x64") else jnp.float64

    key = jax.random.PRNGKey(seed)
    norms = []

    print(f"\n=== Estimating C_W (L={L}, beta={beta}) ===")
    t0 = time.time()

    # Sample configurations at different scales (amplitudes)
    # The maximum norm often occurs at intermediate amplitudes (r ~ 0.5 - 1.5).
    scales = np.linspace(0.1, 1.5, 15)
    samples_per_scale = max(1, n_samples // len(scales))

    for scale in scales:
        current_max = 0.0
        for i in range(samples_per_scale):
            key, subkey, seedkey = jax.random.split(key, 3)

            # Sample random configuration A
            theta = (scale * jax.random.normal(subkey, (n_params,), dtype=dtype))

            # Compute ||H_W(A)|| using the operator norm Lanczos
            norm = lanczos_opnorm(flat_action_grad_W, theta, k=30, seed=int(seedkey[0]))
            norms.append(norm)
            current_max = max(current_max, norm)

        print(f"  Scale r={scale:.2f}: Max ||H_W|| = {current_max:.4f}")

    max_norm = np.max(norms)
    # Normalize by beta to find C_W
    C_W_est = max_norm / beta

    print(f"\nEstimation finished in {time.time()-t0:.2f}s")
    print(f"Maximum ||H_W|| observed: {max_norm:.4f}")
    print(f"Estimated C_W: {C_W_est:.4f}")
    return C_W_est

# =========================================================================
# 6. EXECUTION BLOCK (C_W Estimation)
# =========================================================================

if __name__ == "__main__":
    # Example: L=4, beta=1.0 (Normalization point)
    L_CW = 4
    BETA_CW = 1.0

    # To run the estimation, uncomment the following line:
    # Cw_result = estimate_CW(L=L_CW, beta=BETA_CW, n_samples=50)

    print("\nC_W Estimation Script loaded. Uncomment 'estimate_CW' call to run.")



C_W Estimation Script loaded. Uncomment 'estimate_CW' call to run.
